# Creating my own Voice Assistant using LangGraph and MCP servers

## All locally run, no API keys, cloud services, or subscription fees.

## Why?

I have an Amazon Alexa, but I don't like it. Why? It fails to do anything more complicated than basic voice commands.

I end up using it for three things: 
- Get the current date or time
- Get weather information for today
- Turn on or off connected devices (e.g. TV, lights, robot vacuum)

which are the only things that I can use it for reliably. Anything else, I get a polite and unhelpful "I can't help with that".

Given the rise of LLM Agents and MCP servers, it's become easier than ever to create personal assistants and chatbots. And I ask myself, "Why stop at a chatbot? Why not take this one step further and create my own voice assistant?"

This is my attempt to do just that.



## Goals

So I think, what exactly do I want my voice assistant to be able to do? 

This is my list of initial goals:

### 1. Run on my local computer

I don't want to pay for a subscription to use an LLM, and in fact, I don't want to pay for anything.
Everything I build should just run on my local computer without having to worry about costs or how much free credit I have left at the end of each month.

### 2. Replicate Alexa functionality

Let's take baby steps - first I simply want to replicate the functionality I already have with Alexa. This will be a good milestone to work towards, before I add more complex, extravagant features.

It should be able to:

- Get the current date or time
- Get weather information for today
- Turn on or off connected devices (e.g. TV, lights, robot vacuum)

before we start building this out into a fully-fledged Tony Stark's Jarvis-esque voice assistant that can compute how to travel back in time.

### 3. Be quick

If the responses aren't fast enough, the voice assistant is as good as being silent.

Asking a question and waiting over a minute for a response is unacceptable. I want to be able to ask a question and get a response in a reasonable amount of time. 

However, I know that running anything locally on my cute little Macbook Air is going to be slow, regardless of how many tweaks and refactorings I do.

So for now, I'm not going to expect millisecond-level response times. Instead the response times should be quicker than the time it takes me to execute the task/query myself. At least in this way I know that I'm saving time.

In future articles, we'll delve deeper into the optimisations I do to get this down to millisecond response times without paying for subscriptions.



## My Device Specs

- Macbook Air
- Chip: Apple M3
- Memory: 16GB


## 1. Overall Structure

I've structured the project as follows:

![image](resources/architecture-diagram.png)


### Voice Assistant

#### 1. Speech-to-Text & Text-to-speech

We make use of `RealtimeSTT` for wakeword detection (e.g. "Alexa", "Hey Jarvis", "Hey Siri"), speech detection and real-time transcription.

The transcribed text is then sent to the Agent for processing, after which its response is then streamed to a text-to-speech model. The output is then sent to the speaker.

#### 2. Agent

We use Ollama to run LLMs locally. The agent and the workflow that it takes is implemented in LangGraph.

The agent is responsible for taking a user query, understand it, and call on the tools it thinks are required to provide an appropriate response.

Our voice assistant will require the following tools to meet our goals:

- A function to get the current date.
- A function to get the current time.

It also needs tools to interact with smarthome devices, but the implementation for this can get quite involved so we implement this in a separate MCP server.

#### 3. MCP Server - Smarthome Connection

The MCP server is where we encapsulate the complexity of finding, connecting and managing the devices.

A SQL database keeps track of devices, their connection information and their names. The tools are the interface through which an agent finds the connection information for a given device, and then uses it to turn the device on or off.

## 2. Implementation Details

## Text-to-speech Implementation

The text-to-speech layer was perhaps the easiest to implement. 

Given some string we assume comes from the agent, pass it through a pre-trained text-to-speech model and stream it to the device speaker.

Firstly, let's define a class called `Voice` that will be responsible for this.

We know upfront that apart from the model that we use for speech synthesis, receiving text and streaming it to the speaker will be the same and can remain decoupled from anything model related.

```python
class Voice():
    def __init__(
        self,
        sample_rate: int = 24000,
        chunk_size: int = 2048
    ):
        self.sample_rate = sample_rate
        self.chunk_size = chunk_size
        self.initialise_model()

    def initialise_model(self):
        """Initialise the model to use for TTS."""
        pass

    def convert_text_to_speech(self, text:str) -> list[np.ndarray]:
        """Convert text to sepeech and return the waveform as frames."""
        pass

    def speak(self, text:str):
        """Speak the provided text through device output."""
        frames = self.convert_text_to_speech(self, text)
        for frame in frames:
            self.output_stream.write(frame.tobytes())
```

so we can implement the `speak` function to stream the text to the speaker upfront.

Now, we can figure out which model is out there, which one to use, and how to use it, and then wire it up into our `Voice` class.

## Models Testing

Below, I list the various different TTS models that I experimented with, and the code you can use to replicate the results.

### 1. [BarkModel](https://github.com/suno-ai/bark)


Quickstart code to run the model yourself:

```python
from IPython.display import Audio
from transformers import BarkModel, BarkProcessor

model = BarkModel.from_pretrained("suno/bark-small")
processor = BarkProcessor.from_pretrained("suno/bark-small")
sampling_rate = model.generation_config.sample_rate

input_msg = "The time is 3:10 PM."

inputs = processor(input_msg, voice_preset="v2/en_speaker_2")
speech_output = model.generate(**inputs).cpu().numpy()

Audio(speech_output[0], rate=sampling_rate)
```

Summary:

Good: Very realistic voice synthesis with natural sounding 'umm', 'ahh' filler words. 

Bad: Quality is worse with shorter sentences. The end of the sentence is spoken as though a follow up sentence will quickly follow.

Bad: Very slow. Takes 13 seconds to generate the speech for "The time is 3:10 PM."

### 2. Coqui TTS

Install using:
```bash
pip install coqui-tts
```

Test code:
```python
from TTS.api import TTS 

tts = TTS(model_name="tts_models/en/ljspeech/tacotron2-DDC", progress_bar=False)

output_path = "output.wav"
input_msg = "The time is 3:10 PM."
tts.tts_to_file(text=input_msg, file_path=output_path)
Audio(output_path)
```

Good: Fast. Takes 0.3 seconds to generate the speech for "The time is 3:10 PM."

Bad: Text normalisation is not up to scratch. When it comes to time related queries, the pronunciation of "PM" is off. When the time is set to "13:10 PM", the pronunciation of "13" is unrecognisable.

### 3. Elevenlabs

Install using:

```bash
pip install elevenlabs
```

and run using:

```python
import dotenv
from elevenlabs.client import ElevenLabs
from elevenlabs import stream

dotenv.load_dotenv()

api_key = os.getenv('elevenlabs_apikey')

elevenlabs = ElevenLabs(
  api_key=api_key,
)

audio_stream = elevenlabs.text_to_speech.stream(
    text="The time is 03:47AM",
    voice_id="JBFqnCBsd6RMkjVDRZzb",
    model_id="eleven_flash_v2_5"
)

# option 1: play the streamed audio locally
stream(audio_stream)
```

By far the best in terms of quality and response times, which obviously it should be given it is a paid service. They also provide some free credits without a subsrciption, but I'd rather not become dependent on it at all when developing my voice assistant so we skip it for now.

### 4. Kokoro

We leave the best til last.

[Kokoro](https://github.com/hexgrad/kokoro)

Install using:

```bash
pip install kokoro pyaudio
```

Test code:

```python
RATE = 24000
CHUNK_SIZE = 1024

p = pyaudio.PyAudio()
print(f"Input device: {p.get_default_input_device_info()}")
print(f"Output device: {p.get_default_output_device_info()}")

output_stream = p.open(
    format=pyaudio.paFloat32,
    channels=1,
    rate=RATE,
    output=True,
)
input_msg = "The time is 03:47AM"
generator = pipeline(input_msg, voice='af_heart')
for i, (gs, ps, audio) in enumerate(generator):
    print(i, gs, ps)

    for start in range(0, len(audio), CHUNK_SIZE):
        chunk = audio[start:start + CHUNK_SIZE]
        output_stream.write(chunk.numpy().astype(np.float32).tobytes())
```

Firstly, it's quick - it's on-par with Elevenlabs, only marginally slower, not really noticeable given the example text.
Secondly, the quality of the speech is also good. Sure, it could be better, there are occasions where it sounds slightly clunky.

But on average the quality of the outputs are spot on.

## TTS Implementation

So, we decide to use Kokoro for our text-to-speech implementation. Let's now fill in the blanks for our `Voice` class. Also keep in mind that this is a first implementation, and I know in the future I will want to try other models. 

So instead of implementing the model specific code directly into the `Voice` class, I'll create a child class that inherits from `Voice`. This way, I can do a quick switcharoo between different models without having to change the `Voice` class or unravel code that has become coupled.

```python
from kokoro import KPipeline

class KokoroVoice(Voice):
    def __init__(self, voice:str, sample_rate: int = 24000, chunk_size: int = 2048):
        """Initialise the model to use for TTS.
        
        Args:
            voice (str):
                The voice to use.
                See https://github.com/hexgrad/kokoro/blob/main/kokoro.js/voices/
                for all voices.
            sample_rate (int, optional):
                The sample rate to use. Defaults to 24000.
            chunk_size (int, optional):
                The chunk size to use. Defaults to 2048.
        """
        self.voice = voice
        super().__init__(sample_rate, chunk_size)

    def initialise_model(self):
        """Load the model to use for TTS."""
        self.pipeline = KPipeline(lang_code="b")

    def convert_text_to_speech(self, text:str) -> list[np.ndarray]:
        """Convert text to speech and return the waveform as frames."""
        generator = self.pipeline(text, voice=self.voice)
        frames = []
        for i, (_, _, audio) in enumerate(generator):
            for start in range(0, len(audio), self.chunk_size):
                chunk = audio[start : start + self.chunk_size]
                frames.append(chunk.numpy().astype(np.float32))
        return frames
```

Now, this implementation allows us to simply import and instantiate this class at the point where we receive text from the agent, and stream it to the device speaker using:

```python
text = "Hello world"
voice = KokoroVoice(**kwargs)
voice.speak(text)
```

## SmartHome MCP Server Implementation

This MCP server is dedicated to finding, connecting and managing smarthome devices. It lives in a separate repository, nicely separated from the voice assistant.

At the time of writing, the only smarthome device I have is a Tapo Smart Plug. You can interact with Tapo devices by using the `python-kasa` library.

Our server needs to do the following:

- Given a device name, turn it on or off.
- Discover new devices and add them to the database.
- Update the device database with the latest device information - this includes the name of the device, the IP address and the MAC address.



### 1. Database

Firstly, let's look at how we will store the device information in a SQL database. For simplicity I will choose `duckdb` as the database backend.


#### Devices table

We firstly define the schema for our first (and only) table called `device`. 

```python
# src/smarthome_mcp_server/database.py

import os
import duckdb
from dataclasses import dataclass


@dataclass
class TableSchema:
    name:str
    columns:dict[str, str]
    primary_key:list[str]


def get_device_table_schema():
    return TableSchema(
        name="device",
        columns={
            "device_id" : "VARCHAR",
            "name": "VARCHAR",
            "ip_address": "VARCHAR",
        },
        primary_key=["device_id"],
    )
```

The `device_id` is the primary key, and by definition need to uniquely identify all devices in our home. Each Tapo device has a unique device-id that we can use.

The `name` is what the user would be referencing as the device name. For example, in our case, the Tapo Smart Plug is connected to our living room light, and is named `lights`.

Finally, the `ip_address` column will be the IP Address that is used to connect to the device in order to control it.


#### DB initialisation

We create some helper functions like `get_create_table_if_not_exists_query` and `initialise_database` functions that we can call to invoke the DB initialisation. For brevity, I show just the `initialise_database` function since the former is self-explanatory:

```python
def initialise_database(db_path:os.PathLike) -> duckdb.DuckDBPyConnection:
    """Get the database connection and create the tables if they don't exist."""
    conn = duckdb.connect(db_path)

    # initialise if not exists tables
    conn.execute(
        get_create_table_if_not_exists_query(get_device_table_schema())
    )

    return conn
```

### 2. Device managment

Finally, we implement the code that will be interacting with the devices and updating the database. 

```python
import duckdb
from dotenv import 

class DeviceManager:
    def __init__(self, conn:duckdb.DuckDBPyConnection) -> None:
        self._conn = conn
    
    ...

    async def turn_on_device(self, device_name: str) -> str:
        """Turn on a device.

        Args:
            device_name (str):
                The name of the device to turn on.
        """
        try:
            device = await self._get_device(device_name)
        except DeviceNotFoundError as e:
            logger.exception(e)
            return f"Device {device_name} not found."

        await device.turn_on()
        return f"Device {device_name} turned on."

    async def turn_off_device(self, device_name: str) -> str:
        """Turn off a device.

        Args:
            device_name (str):
                The name of the device to turn off.
        """
        try:
            device = await self._get_device(device_name)
        except DeviceNotFoundError as e:
            logger.exception(e)
            return f"Device {device_name} not found."

        await device.turn_off()
        return f"Device {device_name} turned off."

    async def list_devices(self) -> list[str]:
        """List the available device names.

        Returns:
            list[str]:
                A list of device names.
        """
        results = self._conn.query("SELECT name FROM device").fetchall()

        return [result[0] for result in results]
```

The three methods above will be the public methods that we register as tools for our Voice Assistant.

We've omitted the private methods for brevity.

One things that I've realised since writing this code is that `DeviceManager` is very `Tapo` specific. I've been naive in thinking that other smart device APIs would follow the same pattern having looked at integrating non-Tapo devices - so in the future, this class will need to be changed to `TapoDeviceManager`, and further abstractions will need to be made to support other APIs.

For example, recently I've got some Wiz lightbulbs for my bedroom. Turns out, the API does not fetch the names assigned to each device via the app, which was available in Tapo by default. Therefore, I will need to think of some way to fetch this in the backend, or use the voice-assistant to populate it when it doesn't exist.


### 3. Expose the tools to Voice-Assistant using FastMCP

Finally, we need to expose the methods we've written as tools for our voice assistant to use. 

```python
from fastmcp import FastMCP

def register_device_manager_tools(mcp_instance: FastMCP, device_manager: DeviceManager) -> FastMCP:
    """Register the methods defined in DeviceManager as tools for MCP server."""
    mcp_instance.tool(name_or_fn=device_manager.list_devices)
    mcp_instance.tool(name_or_fn=device_manager.turn_off_device)
    mcp_instance.tool(name_or_fn=device_manager.turn_on_device)
    return mcp_instance


async def populate_database(device_manager: DeviceManager):
    """Find all devices that are available and update the database.

    Discover all available devices and get their latest states.

    Note:
        Device names may have changed via the mobile app, thus this
        step is necessary when starting the server.
    """
    all_devices = await device_manager.discover_new_devices()
    upsert_coroutines = [device_manager._upsert_device(device) for device in all_devices.values()]
    await asyncio.gather(*upsert_coroutines)


def initialise_server(db_path: os.PathLike) -> FastMCP:
    """Initialise the server.

    Args:
        db_path (os.PathLike):
            The path to the duckdb database which
            stores the server information.
    Returns:
        FastMCP: The FastMCP server.
    """
    conn = initialise_database(db_path)
    device_manager = DeviceManager(conn)

    # find all devices that are available and update the database
    asyncio.run(populate_database(device_manager))

    mcp = FastMCP(
        name="smarthome-mcp-server",
        instructions="This server is for finding and controlling smarthome devices.",
    )

    register_device_manager_tools(mcp, device_manager)
    return mcp
```

Now, I know there are some holes in the implementation - it's a first attempt and an ongoing project, so if you see any issues or improvements please let me know.

### Server Entry Point

We use `typer` to make our server into a CLI app.

```python
# __main__.py

load_dotenv()

app = typer.Typer()
console = Console()


@app.command()
def main():
    config = load_config()

    # set up server data directory
    root_dir = platformdirs.user_data_path(
        appname="smarthome-mcp-server",
        ensure_exists=True
    )
    db_path = Path(root_dir) / config.database.path
    db_path.parent.mkdir(parents=True, exist_ok=True)
    logger.info("Server data directory: %s", db_path)

    # init and run
    mcp_instance = initialise_server(db_path)
    asyncio.run(mcp_instance.run_stdio_async())

if __name__ == "__main__":
    app()
```

We run the server `python -m smarthome_mcp_server`:

```bash

╭─ FastMCP 2.0 ────────────────────────────────────────────────────────────╮
│                                                                          │
│        _ __ ___ ______           __  __  _____________    ____           │
│    ____                                                                  │
│       _ __ ___ / ____/___ ______/ /_/  |/  / ____/ __ \  |___ \  / __    │
│    \                                                                     │
│      _ __ ___ / /_  / __ `/ ___/ __/ /|_/ / /   / /_/ /  ___/ / / / /    │
│    /                                                                     │
│     _ __ ___ / __/ / /_/ (__  ) /_/ /  / / /___/ ____/  /  __/_/ /_/     │
│    /                                                                     │
│    _ __ ___ /_/    \__,_/____/\__/_/  /_/\____/_/      /_____(_)____/    │
│                                                                          │
│                                                                          │
│                                                                          │
│    🖥️  Server name:     smarthome-mcp-server                              │
│    📦 Transport:       STDIO                                             │
│                                                                          │
│    📚 Docs:            https://gofastmcp.com                             │
│    🚀 Deploy:          https://fastmcp.cloud                             │
│                                                                          │
│    🏎️  FastMCP version: 2.11.2                                            │
│    🤝 MCP version:     1.12.4                                            │
│                                                                          │
╰──────────────────────────────────────────────────────────────────────────╯


[08/19/25 05:02:55] INFO     Starting MCP server              server.py:1445
                             'smarthome-mcp-server' with                    
                             transport 'stdio'                              
```


### 4. Using the tools

Now that the server has been implemented, we can now define some methods that will interact with the server via a client. This client will be used to register the tools for the Voice Assistant to use.

Coming back to the voice-assistant repo:

```python
from langchain_mcp_adapters.client import MultiServerMCPClient

def get_new_mcp_client() -> MultiServerMCPClient
    return MultiServerMCPClient(
        {
            "smarthome-mcp-server": {
                "command": "smarthome_mcp_server",
                "args": [],
                "transport": "stdio",
            }
        }
    )
```

This method uses the convenient `MultiServerMCPClient` class to register our smarthome MCP server for tool usage. 

The returned client object then exposes a `get_tools` method which returns all the tools that the registered servers expose.

```python
mcp_client = get_new_mcp_client()
tools = await mcp_client.get_tools()
```

Note how we use `await` here given the `get_tools` method is asynchronous.

By defining a function called `get_mcp_server_tools`:

```python
def get_mcp_server_tools():
    mcp_client = get_new_mcp_client()
    tools = await mcp_client.get_tools()
    return tools
```

this single function can be imported into wherever we define our agent and register the tools for use.

## Speech-to-text Implementation

The speech-to-text (STT) is where a lot of complexity comes in as it requires realtime IO processing.

STT itself is simple enough to achieve - there are plenty of models out there that we can use. But we also need to be able to constantly listen for a user's voice input, which consists of a wakeword and a query.

A wakeword is what you normally use to trigger a voice assistant to start listening to you. For example, "Hey Google" or "Hey Siri", or "Alexa".

I could write this code entirely myself, but to make things simpelr, I had a quick dig around just in case there was something pre-built that I could use. And to my surprise, I found the package [`RealtimeSTT`](https://github.com/KoljaB/RealtimeSTT) and it works perfectly.


### How it works in a nutshell

1. Create a thread for listening to the user's voice input. Another for transcribing, which runs the STT model.
2. If a wakeword is detected, start recording the user's voice input.
3. The recorded audio is then sent to the STT model for transcribing, and returns the transcribed text as a string.

To use this package, all we need to do is use the `AudioToTextRecorder` class as a context manager like below:

```python
from RealtimeSTT import AudioToTextRecorder

with AudioToTextRecorder(
    model='tiny',
    wakeword_backend='oww',
    wake_words='hey jarvis',
    device='cpu',
    wake_word_activation_delay=3.0,
    wake_word_buffer_duration=0.15,
    post_speech_silence_duration=1.0
) as recorder:
    while True:
        # get the transcribed text from recorder
        query = recorder.text()
        if (query is not None) and (query != ""):

            # get response from our langgraph agent
            response_stream = await get_response_stream(
                query, agent_executor, thread_config
            )

            # output the response to device audio
            await stream_voice(response_stream, output_chunk_builder, voice)
```

We will come back to `get_response_stream` and `stream_voice` methods in the next section, since this involves how we define our agent.

But simply putting together the `AudioToTextRecorder` context manager in the way we have, we've got a working speech -> text -> response mechanism implemented. 

If you were to simply replace the `get_response_stream` with any LLM agent, and replace the `stream_voice` with any text-to-speech agent, you would have a working voice assistant. You could also use a simple `print` statement and you would have a rudimentary chat bot with voice input.

## Agent Implementation

Finally, the good stuff - the agent implementation. 

I've left this as last since it's a bit more involved. Let's get stuck in.

### LangGraph - What is it? 

LangGraph is a framework for building stateful, graph-based workflows with language model agents.

Nodes encapsulate any logic related to an action an LLM agent can take.

Edges encapsulate the logic which determines how to transition from one node to another.

`Langgraph` implements a prebuilt graph that we can get via the `create_react_agent` method. The graph looks like this:

![prebuilt-graph](resources/prebuilt-graph.png)

Let's use this as an example to explain better how nodes and edges work.

As you can see, the graph is very simple:

- Given a query (the `__start__` node)
- The `agent` will receive the query and determine whether it needs to call a tool to be able to respond appropriately. If it does, we transition to the `tool` node. Once the tool response is received, we go back to the `agent` node. The agent will repeatedly call the appropriate tools until it determines it has everything it needs.
- Then, it will return its response (the `__end__` node)

The conditional transition between the `agent`, `tools` and `__end__` node is represented as dashed lines. Then, the question is: How do we determine which node to go to next?

Well, Langgraph maintains a log of the messages that have been sent, and this represents the state of the graph. The messages can come from the user, the agent, or a tool. In this example, the `agent` node will create a message that explicitly states that it will call a tool (exactly how will be revealed in the next section). The presence of this tool call is what triggers the transition from the `agent` node to the `tools` node.

If no tools are called, then the transition from the `agent` node to the `__end__` node is triggered.

It is this check for the presence of tool calls that is implemented in the conditional edge between the `agent`, `tools` and `__end__` nodes.

In a future article, I'll go into an example of how I created a custom agent graph to optimise for latency, and demonstrate how exactly these conditional edges and nodes are implemented. For now, we don't need to go into too much detail about this since the prebuilt graph is good enough for the scope of this article. 

### Our Agent Implementation

So, we define a function called `get_new_agent` like below:

```python
from langgraph.prebuilt import create_react_agent
from langgraph.graph.state import CompiledStateGraph

from voice_assistant.tools.datetime import get_tools as get_datetime_tools


def get_new_agent(
    config, short_term_memory, long_term_memory
) -> CompiledStateGraph:
    """Build and return a new graph that defines the agent workflow."""
    
    # initialise the LLM
    model = init_chat_model(
        model=config.Agent.model,
        model_provider=config.Agent.model_provider,
        temperature=0,
        reasoning=config.Agent.reasoning
    )

    # initialise the tools that the agent will use
    server_tools = await get_mcp_server_tools()

    tools = (
        get_datetime_tools()
        + server_tools
    )

    # build the agent workflow given the LLM, its tools and memory.
    agent_executor = create_react_agent(
        model,
        tools,
        checkpointer=short_term_memory,
        store=long_term_memory
    )

    return agent_executor
```

which is responsible for:

1. initialise the LLM - `init_chat_model` returns the LLM from the specified provider. In our case, we use Ollama as our provider, and `llama3.2:latest` as our model type.

2. Defining the full set of tools that the agent will use. We have a function called `get_datetime_tools()` which returns a `list` of `StructuredTool` objects. We also have `server_tools`, which are the list of tools that our previously mentioned MCP server provides for home automation. Additionally, If we wish to extend the set of tools the agent can use, this is the place to add them.

3. Construct the agent workflow given the LLM and its tools. Here is where we call the `create_react_agent` function from LangGraph. The function can also take in `checkpointer` and `store` objects which are used to persist the state of the agent, acting as a short term and long term memory.

In the future, if we want to use a custom graph, we can replace the `create_react_agent` function call with our own implementation.

### Handling the Agent Response

Now, we've so far implemented all the components that we need to

1. get the user query
2. get the tools
3. create the agent

The next step is to run the agent to get a response for the query, and output it via the `Voice` method we defined earlier.

Given the user query text that we've received from our STT implementation, we format it into a dictionary:

```python
user_query = "Hello world!"
user_query_formatted = {
    "role": "user",
    "content": user_query
}
```

This dictionary tells the agent that the message is from the user.

We also add a system prompt to set the context and give instructions to the agent:

```python
system_prompt_formatted = {
    "role": "system",
    "content": (
        "You are a voice assistant called Jarvis."
        + " Keep your responses as short as possible."
        + "Do not format your responses using markdown, such as **bold** or _italics. ",
    )
}
```

These two messages are then passed into the agent to get a response:

```python
response = agent_executor.invoke(
    {"messages" : [system_prompt_formatted, user_query_formatted]},
)
```

The response is a dictionary of messages:

```python
output
> {
    "messages": [
        SystemMessage(
            content="You are a voice assistant called Jarvis.Keep your responses as short as possible.Do not format your responses using markdown, such as **bold** or _italics. ",
            additional_kwargs={},
            ...
        ),
        HumanMessage(
            content="What time is it?",
            additional_kwargs={},
            ...
        ),
        AIMessage(
            content="",
            additional_kwargs={},
            tool_calls=[
                {
                    "name": "get_current_time",
                    "args": {},
                    "id": "b39f7b12-4fba-494a-914a-9d4eaf3dc7d1",
                    "type": "tool_call",
                }
            ],
            ...
        ),
        ToolMessage(
            content="11:32PM",
            name="get_current_time",
            ...
        ),
        AIMessage(
            content="It's currently 11:32 PM.",
            additional_kwargs={},
            ...
        ),
    ]
}
```

As you can see, the output is a list of all the messages that have been created throughout the graph execution.

The first message will always be a `HumanMessage` or a `SystemMessage` since this is what we provided to the agent as input (i.e. the `__start__` node). 

The remaining are the messages that the agent or tools returned, in the order they were called.

For example, you can see the first `AIMessage`, the message type generated by the LLM, has a tool call within it which uses a `get_current_time` tool. The presence of a `tool_calls` property in the `AIMessage` is what triggers the conditional transition from the `agent` node to the `tools` node.

![agent-response](resources/prebuilt-graph-with-agent-tool-edge-highlighted.png)

Then you see the `ToolMessage` which is the response that was returned by the `get_current_time` tool. 

Finally, the model responds with the actual response to the user query. The lack of a `tool_calls` property in the `AIMessage` means that the graph should transition to the `__end__` node and return the response.

### Reducing latency

The issue with the above is that we are waiting for the entire agent workflow to complete before we get a response. This can take a long time if the agent is addressing a complex query.

Therefore, to improve on this, we can use the `stream` mode in `langgraph` to stream the response as they are generated. This allows us to start voicing the response as they are generated, rather than waiting for the entire response to be generated and then voicing it all in one go.

```python
output_stream = agent_executor.stream(
    {"messages" : [system_prompt_formatted, user_query_formatted]},
    stream_mode="messages"
)
```

Here, `output_stream` is a generator that will yield a tuple of messages and message metadata, as they come.

Note, there is an asynchronous version of this method called `astream`, which does exactly the same thing but returns an `AsyncIterator` instead. 

If we look at the messages we get after this change:

```python
print([chunk for chunk, metadata in output])

>   AIMessageChunk(
        content="",
        tool_calls=[{"name": "get_current_time", ...}],
        tool_call_chunks=[{"name": "get_current_time", "args": "{}", ...}],
    ),
    ToolMessage(content="01:21AM", name="get_current_time", ...),
    AIMessageChunk(content="It", ...),
    AIMessageChunk(content="'s", additional_kwargs={}, ...),
    AIMessageChunk(content=" currently", ...),
    AIMessageChunk(content=" ",), 
    AIMessageChunk(content="1", ...), 
    AIMessageChunk(content=":", ...), 
    AIMessageChunk(content="21", ...),
    AIMessageChunk(content=" AM", ...),
    AIMessageChunk(content=".", ...),
    AIMessageChunk(content="", ...),
```

You can now see the tokens are being returned as they are generated.

But this poses a new problem: we can't just give the TTS model individual tokens - it will just pronounce each token one by one, i.e. `"It"`, `"'s"` will be pronounced separately, which is definitely not what we want.

So, while we need to stream the repsonse for latency reasons, there is still a need to wait and accumulate tokens and group them into meanigful chunks, before sending them to the TTS model.

### Building Output Chunks

We therefore handle this complexity by defining an `OutputChunkBuilder`. So what constitutes a _meaningful_ chunk?

The first thing that comes to mind is to wait until a full sentence, i.e. append all the tokens until it ends with one of `.`, `?`, `;`, `!`.

From trial and error, it has also proven wise to include `\n` in this list as well, when we get a particularly long response from the agent.

```python
class OutputChunkBuilder:
    def __init__(self):
        self._msg = ""
        self.end_of_sentence = (".", "?", ";", "!", "\n")

    def add_chunk(self, message_chunk:str):
        self._msg += message_chunk

    def output_chunk_ready(self) -> bool:
        return self._msg.endswith(self.end_of_sentence)
```

We've implemented this logic above, one function that appends message chunks together into a buffer called `_msg`, and one to check if the collated messages are ready (i.e. a full sentence or a new line).

```python
class OutputChunkBuilder:
    
    ... # omitted for brevity

    def _reset_message(self):
        self._msg = ""

    def get_output_chunk(self):
        msg = self._msg # Get the current message chunk
        self._reset_message()
        return msg
```

We also implement the `get_output_chunk` function which will return the messages collated so far, and also reset the message buffer to an empty string so that it is ready for collating the next set of chunks.

This enables us to use logic like below to stream the response, sentence by sentence:

```python
def stream_voice(msg_stream, output_chunk_builder):
    for chunk, metadata in msg_stream:
        # append the chunk to our buffer
        if chunk.content != "":
            output_chunk_builder.add_chunk(chunk.content)

        # speak the output chunk if it is ready
        if output_chunk_builder.output_chunk_ready():
            voice.speak(output_chunk_builder.get_output_chunk())
```

## Tools Implementation

Lastly, let's look at how we can implement the tools required to get the current date and time.

This is very straightforward, by far the easiest implementation. Any function that you create can be used as a tool as long as the docstrings are well-written and formatted clearly.

There are two ways to mark a function as a tool:

1. Using the `@tool` decorator from `langchain_core.tools`
2. Using the `StructuredTool` class from `langchain_core.tools.structured`

For easier unit testing of our tools, we opt for the second option since the first option does not allow us to import the tool function into our tests.

Additionally, we write a simple wrapper function called `get_now_datetime` that returns the current datetime, which also makes it easier to mock in our unit tests.

Then, we are able to define the `get_current_time` function as follows:



```python
# tools/datetime.py

from datetime import datetime
from langchain_core.tools.structured import StructuredTool


def get_now_datetime() -> datetime:
    """Wrapper for easier mocking in unit test."""
    return datetime.now()

def get_current_time() -> str:
    """Get the current time in format HH:MM AM/PM"""
    return get_now_datetime().strftime("%I:%M%p")
```

Next, a function for getting the current date. 

We have to be careful here - different text-to-speech (TTS) models have varying abilities when it comes to text normalisation. 

e.g. if the function `get_current_date` returns the string `01-01-2025`, the TTS model may pronounce this as 'oh one oh one twenty twenty five'.

To make our implementation robust to differences in model capabilities, we normalise the date string to be clearer in how the date should be pronounced:

```python
def _convert_date_to_words(dt: datetime):
    """Change date values represented in YYYY-mm-dd format to word values as they would be pronounced."""
    day = dt.day
    if day == 1 or day == 21 or day == 31:
        day_word = f"{day}st"
    elif day == 2 or day == 22:
        day_word = f"{day}nd"
    elif day == 3 or day == 23:
        day_word = f"{day}rd"
    else:
        day_word = f"{day}th"

    date_obj = dt.strftime(f"%B {day_word}, %Y")
    return date_obj

def get_current_date() -> str:
    """Get the current date in format YYYY-MM-DD"""
    dt = get_now_datetime()
    dt_str = _convert_date_to_words(dt)
    return dt_str
```

By defining a `_convert_date_to_words` function, we now convert a `datetime` object like `datetime(2025, 1, 1)` into `January 1st, 2025`.

Finally, we write a `get_tools` function which will wrap up the `get_current_time` and `get_current_date` methods into a `StructuredTool`, and return them in a list:

```python
def get_tools():
    """Get a list of tools for the agent.

    Returns:
        A list of tool functions available to the agent.
    """
    return [
        StructuredTool.from_function(get_current_time),
        StructuredTool.from_function(get_current_date),
    ]
```

thereby allowing us to import this function and callling it when we create the agent, as we saw in the agent implementation section.

## Putting it all together

Now, we've gone through the individual components that make up our voice assistant, time to assemble them together.

```python
# main.py

from RealtimeSTT import AudioToTextRecorder
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.store.sqlite.aio import AsyncSqliteStore

from voice_assistant.agent import get_new_agent, get_response_stream
from voice_assistant.voice import KokoroVoice
from settings import load_config


async def main():

    conf = load_config()
    voice = KokoroVoice(**conf.KokoroVoice)
    output_chunk_builder = OutputChunkBuilder()
    thread_config = {"configurable": {"thread_id": "abc123"}}

    # short term memory
    async with AsyncSqliteSaver.from_conn_string(conf.Agent.memory.checkpointer) as saver:
            
            # long term memory
            async with AsyncSqliteStore.from_conn_string(conf.Agent.memory.store) as store:
                
                agent_executor = await get_new_agent(conf, saver, store)

                with AudioToTextRecorder(**conf.AudioToTextRecorder) as recorder:
                    while True:
                        query = recorder.text()
                        if (query is not None) and (query != ""):
                            response_stream = await get_response_stream(
                                query, agent_executor, thread_config
                            )
                            await stream_voice(response_stream, output_chunk_builder, voice)


if __name__ == "__main__":
    asyncio.run(main())
```

Here, we load in our Yaml config file using `OmegaConf`:

```python
# settings.py

import logging
from pathlib import Path
from omegaconf import OmegaConf


logger = logging.getLogger(__name__)


CONFIG_PATH = Path(__file__).parents[1] / "conf" / "config.yaml"


def load_config():
    logger.debug(f"Loading config from: {CONFIG_PATH}")
    return OmegaConf.load(CONFIG_PATH)
```

and use SQL databases to store our short and long term memory - this is done using the `AsyncSqliteSaver` and `AsyncSqliteStore` classes from the `checkpoint` and `store` modules in `langgraph`.

In a while loop, the STT thread records the user's voice input after a wakeword is detected, which is then passed to the agent for processing. The agent response is returned as an AsyncIterator, which we then stream to the device speakers using the `stream_voice` function.

The `stream_voice` function looks like this:

```python
async def stream_voice(
    msg_stream: AsyncGenerator,
    output_chunk_builder: OutputChunkBuilder,
    voice: Voice
):
    """Stream messages from the agent to the voice output."""
    async for chunk, metadata in msg_stream:
        if metadata["langgraph_node"] == "agent":
            # build up message chunks until a full sentence is received.
            if chunk.content != "":
                output_chunk_builder.add_chunk(chunk.content)

            if output_chunk_builder.output_chunk_ready():
                voice.speak(output_chunk_builder.get_output_chunk())

    # if we have anything left in the buffer, speak it.
    if output_chunk_builder.current_message_length() > 0:
        voice.speak(output_chunk_builder.get_output_chunk())
```

Which is the same logic as we already discussed before in the `Building Output Chunks` section, but with some small tweaks.

It turns out, not all responses end with a puctuation mark, e.g. when the LLM uses bullet points in their response, I've found they omit the punctuation for each bullet point.

So, we make sure to flush our buffer at the end if it isn't empty.

We also filter out any messages that are not from the agent, as we don't want to stream the user's input or the tool responses back to the device speakers. We do this by checking the `langgraph_node` metadata key, and only speaking the message if it comes from the agent.

And turns out, that's all you need to build a fully functioning voice assistant.

## Final Remarks

Overall, I've been pleasantly surprised at how easy it was to build this out. 

Sure, there are certainly more optimisations that can be made, but given I've been able to get the full functionality working within two weeks (whilst working a full-time job), I'm happy with the results.

But we're not done yet. 

There are a whole load of things I've had to keep out of this article for the sake of brevity, including the additional optimisations I've had to make to make the voice assistant quicker, so this will be covered in my next article.

The full project can be found here: 

https://github.com/bl3e967/voice-assistant

If you wish to engage with me, then take a look at my Patreon page: patreon.com/BenjaminLeeDataScience 

where you can interact with me directly, whether you want to discuss this article, or ask me a question, or even influence the direction of future articles and projects. 

All of my articles can be found here: https://medium.com/@bl3e967